
# Scientist-in-the-Loop: Generating Semantic Graph Weights using Vision LLMs
This notebook handles the collection of visual surface data (satellite and ground imagery) and uses a local Vision Language Model to analyze surface roughness and micrometeorological characteristics. These text descriptions are then converted into custom graph edge weights.


In [ ]:

import os
import glob
import json
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm



## 1. Visual Data Collection
Downloads top-down satellite imagery from NASA GIBS and directional ground photos from CIMIS for all target stations.


In [ ]:

def download_satellite_image(lat, lon, station_id, zoom=8, size=512, date="2024-07-01", save_dir="data/satellite"):
    """Downloads top-down satellite view from NASA GIBS."""
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    base_url = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"
    delta = 0.5 / zoom 
    bbox = f"{lon - delta},{lat - delta},{lon + delta},{lat + delta}"

    params = {
        "SERVICE": "WMS", "VERSION": "1.3.0", "REQUEST": "GetMap",
        "LAYERS": "MODIS_Terra_CorrectedReflectance_TrueColor",
        "TIME": date, "CRS": "EPSG:4326", "BBOX": bbox,
        "WIDTH": size, "HEIGHT": size, "FORMAT": "image/jpeg"
    }

    try:
        response = requests.get(base_url, params=params, timeout=15)
        if "image" in response.headers.get("Content-Type", "").lower():
            img = Image.open(BytesIO(response.content))
            filename = f"station_{station_id}_satellite.jpg"
            img.save(os.path.join(save_dir, filename))
            return True
    except Exception as e:
        print(f"Satellite error for station {station_id}: {e}")
    return False

def download_cimis_site_photos(station_id, save_dir="data/station_images"):
    """Downloads ground-level directional photos."""
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    direction_map = {'North': 'Nt', 'South': 'St', 'East': 'Et', 'West': 'Wt'}
    padded_id = str(station_id).zfill(3)
    base_url = "https://cimis.water.ca.gov/App_Themes/Images/Stations"

    downloaded = 0
    for direction_name, suffix in direction_map.items():
        img_name = f"{padded_id}{suffix}.jpg"
        url = f"{base_url}/{img_name}"
        
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                local_name = f"{station_id}_{direction_name}.jpg"
                with open(os.path.join(save_dir, local_name), 'wb') as f:
                    f.write(response.content)
                downloaded += 1
        except Exception:
            pass
    return downloaded


In [ ]:

# --- BATCH IMAGE DOWNLOAD EXECUTION ---
# Load registry from Data Collection step
registry_df = pd.read_csv("data/station_registry.csv")

print(f"🚀 Starting Visual Data Download for {len(registry_df)} stations...")

total_sat = 0
total_ground = 0

for _, row in tqdm(registry_df.iterrows(), total=len(registry_df)):
    # NASA Satellite
    if download_satellite_image(row['Lat'], row['Lon'], row['ID']):
        total_sat += 1
        
    # CIMIS Ground
    total_ground += download_cimis_site_photos(row['ID'])

print(f"✅ Success! Downloaded {total_sat} Satellite images and {total_ground} Ground photos.")



## 2. LLM Prompt Configuration
Defines the system prompts for the micrometeorologist persona. These prompts constrain the LLM to focus strictly on physical surface features rather than making generic weather speculations.


In [ ]:

SATELLITE_PROMPT = """
Role: You are a professional meteorologist and land–atmosphere interaction expert.

Task: Analyze satellite imagery strictly as a surface characterization product for numerical weather prediction (NWP) and mesoscale modeling.

Constraints: 
- Description must be based ONLY on visual surface features. 
- Avoid speculation about atmospheric conditions not directly inferred from the surface. 
- Use physically cautious language.

Focus Areas:
- Surface roughness and implications for aerodynamic drag
- Land use / land cover heterogeneity
- Topography, terrain gradients, and orographic influence
- Spatial variability of surface albedo
- Land surface temperature patterns (relative, not absolute)
- Soil moisture indicators and surface wetness signals
- Vegetation state, density, and phenological stage
- Hydrological features (open water, rivers, wetlands, snow, ice)
- Urban morphology and built-up surface characteristics
- Surface energy balance implications and flux partitioning tendencies
- Terrain-induced or surface-driven mesoscale circulations
- Relevance for lower boundary condition specification in NWP models

Language Directive: Avoid speculative weather statements. Use neutral, inference-based phrasing such as "suggests", "is consistent with", "may indicate", "implies".

Format: Output a short, dense scientific paragraph suitable for model preprocessing documentation.
"""

def get_station_prompt(direction):
    return f"""
Role: You are a professional meteorologist specializing in micrometeorology and land–atmosphere interactions.

Task: Analyze a ground-level photograph representing one directional view ({direction}) of a meteorological station's immediate environment as a LOCAL surface characterization product.

Constraint: Base analysis ONLY on directly observable surface features at this local scale. Do NOT infer atmospheric conditions, mesoscale processes, or regional properties.

Focus Areas (where visually supported):
- Local surface roughness elements (vegetation, buildings, obstacles) and their implications for aerodynamic drag and turbulence generation
- Immediate land use / land cover type and heterogeneity within the station footprint
- Local terrain slope, elevation changes, or obstacles influencing flow exposure and sheltering
- Surface material characteristics relevant to reflectance and heat storage (e.g., soil, grass, asphalt, concrete)
- Visual indicators of soil moisture or surface wetness (e.g., bare soil, vegetation condition, water presence)
- Vegetation type, density, height, and phenological state
- Presence of nearby hydrological features (ponds, channels, irrigation, snow, ice)
- Built environment elements affecting radiative and aerodynamic conditions (walls, roads, fences, urban structures)
- Implications for surface energy exchange and representativeness of the station measurements

Language Directive: Use physically cautious phrasing such as "suggests", "is consistent with", "may indicate", "implies".  
Avoid references to regional/mesoscale circulation, land surface temperature fields, atmospheric stability, or weather conditions.

Format: Output one concise scientific paragraph suitable for station siting documentation or observational metadata.
"""



## 3. Vision Model Inference
This block processes the downloaded images through the local LLM. 
**Note:** Ensure you have either `Ollama` or `LM Studio` running locally with the selected model loaded.


In [ ]:

# --- CHOOSE YOUR INFERENCE ENGINE ---
# Set to 'OLLAMA' or 'LM_STUDIO'
ENGINE = "OLLAMA"  

if ENGINE == "OLLAMA":
    import ollama
    MODEL_ID = "llava:13b"
    
    def process_image(image_path, prompt):
        station_id = os.path.basename(image_path).split("_")[0]
        p_type = "satellite" if "satellite" in image_path.lower() else next((d.lower() for d in ["North", "South", "East", "West"] if d.lower() in image_path.lower()), "general")
        try:
            response = ollama.chat(
                model=MODEL_ID,
                messages=[{'role': 'user', 'content': prompt, 'images': [image_path]}]
            )
            return station_id, p_type, response['message']['content']
        except Exception as e:
            return station_id, p_type, f"Error: {str(e)}"

elif ENGINE == "LM_STUDIO":
    import lmstudio as lms
    MODEL_ID = "qwen2-vl-7b-instruct"
    vision_model = lms.llm(MODEL_ID)
    
    def process_image(image_path, prompt):
        station_id = os.path.basename(image_path).split("_")[0]
        p_type = "satellite" if "satellite" in image_path.lower() else next((d.lower() for d in ["North", "South", "East", "West"] if d.lower() in image_path.lower()), "general")
        try:
            image_handle = lms.prepare_image(image_path)
            chat = lms.Chat()
            chat.add_user_message(prompt, images=[image_handle])
            response = vision_model.respond(chat)
            return station_id, p_type, response.content
        except Exception as e:
            return station_id, p_type, f"Error: {str(e)}"


In [ ]:

# --- BATCH INFERENCE EXECUTION ---
SATELLITE_DIR = "data/satellite"
STATION_DIR = "data/station_images"
OUTPUT_FILE = "data/station_descriptions.json"

# Load existing JSON to allow resuming
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
else:
    data = {}

# Gather tasks
tasks = []
# Satellite tasks
for img in glob.glob(os.path.join(SATELLITE_DIR, "*")):
    sid = os.path.basename(img).split("_")[0]
    if sid not in data or "satellite" not in data.get(sid, {}):
        tasks.append((img, SATELLITE_PROMPT))

# Station tasks
for img in glob.glob(os.path.join(STATION_DIR, "*")):
    sid = os.path.basename(img).split("_")[0]
    direction = next((d for d in ["North", "South", "East", "West"] if d.lower() in img.lower()), None)
    if direction:
        if sid not in data: data[sid] = {}
        if isinstance(data[sid], str): data[sid] = {"satellite": data[sid]}
        if direction.lower() not in data[sid]:
            tasks.append((img, get_station_prompt(direction)))

print(f"Processing {len(tasks)} pending images using {ENGINE} ({MODEL_ID})...")

# Process sequentially to prevent VRAM overflow
with ThreadPoolExecutor(max_workers=1) as executor:
    futures = [executor.submit(process_image, img, p) for img, p in tasks]
    
    for future in tqdm(as_completed(futures), total=len(futures)):
        sid, ptype, text = future.result()
        if text:
            if sid not in data: data[sid] = {}
            if isinstance(data[sid], str): data[sid] = {"satellite": data[sid]}
            data[sid][ptype] = text
            
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)

print("✅ Scientific Analysis Complete. Output saved to station_descriptions.json")
